# Cockpit

Soma de demanda por origem de material, a partir de uma data parametrizada até a última data disponível.


In [0]:
%sql
WITH demanda AS (
  SELECT
    cad.empresa,
    cad.origem,
    CASE cad.origem
      WHEN '1' THEN 'Fornecedor São Paulo'
      WHEN '2' THEN 'Fábrica HDA'
      WHEN '3' THEN 'Importado'
      WHEN '4' THEN 'Fornecedor Manaus'
      WHEN '5' THEN 'Fábrica HAB'
      ELSE cad.origem
    END AS descricao_origem,
    cad.grupo_de_compradores,
    SUM(d.quantidade) AS total_demanda
  FROM parts_hdbk_sandbox.pr_demand.demand_analytical_base d
  -- View com item_principal_cadeia tratado (SIM/nulo -> material)
  INNER JOIN parts_hdbk_sandbox._agents_databases.material_cadastrao cad
    ON cad.empresa = d.organizacao_vendas
    AND cad.item_principal_cadeia = d.item_principal_cadeia
  WHERE d.data_ordem >= :data_inicio
    AND cad.empresa = '0200'
    AND (
      (cad.origem = '1' AND cad.grupo_de_compradores IN ('042', '043'))
      OR cad.item_principal_cadeia IN ('H0615KRMC00', '06121KVS305', 'H0612KVS305', 'H0615KWG000', '06121K62305')
    )
  GROUP BY cad.empresa, cad.origem, 3, cad.grupo_de_compradores
)
SELECT empresa, origem, descricao_origem, grupo_de_compradores, total_demanda
FROM demanda

ORDER BY empresa, origem, grupo_de_compradores

In [0]:
%sql
WITH forecast AS (
  SELECT
    cad.empresa,
    cad.origem,
    CASE cad.origem
      WHEN '1' THEN 'Fornecedor São Paulo'
      WHEN '2' THEN 'Fábrica HDA'
      WHEN '3' THEN 'Importado'
      WHEN '4' THEN 'Fornecedor Manaus'
      WHEN '5' THEN 'Fábrica HAB'
      ELSE cad.origem
    END AS descricao_origem,
    cad.grupo_de_compradores,
    ROUND(SUM(f.forecast_qty), 0) AS total_forecast
  FROM parts_hdbk_sandbox.pr_forecast.refined_forecast_enriched f
  -- View com item_principal_cadeia tratado (SIM/nulo -> material)
  INNER JOIN parts_hdbk_sandbox._agents_databases.material_cadastrao cad
    ON cad.empresa = f.segment
    AND cad.item_principal_cadeia = f.material
  WHERE f.forecast_cycle = (SELECT MAX(forecast_cycle) FROM parts_hdbk_sandbox.pr_forecast.refined_forecast_enriched)
    AND f.forecast_month >= try_cast(:data_inicio_forecast AS DATE)
    AND f.forecast_month <= try_cast(:data_fim AS DATE)
    AND cad.empresa = '0200'
    AND (
      (cad.origem = '1' AND cad.grupo_de_compradores IN ('042', '043'))
      OR cad.item_principal_cadeia IN ('H0615KRMC00', '06121KVS305', 'H0612KVS305', 'H0615KWG000', '06121K62305')
    )
  GROUP BY cad.empresa, cad.origem, 3, cad.grupo_de_compradores
)
SELECT empresa, origem, descricao_origem, grupo_de_compradores, total_forecast
FROM forecast

ORDER BY empresa, origem, grupo_de_compradores